In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv("../prices.txt", sep=r"\s+")

returns = prices.pct_change()

assets = list(prices.columns)

x = 3






0           NaN
1     -0.015100
2     -0.006701
3      0.001942
4      0.011324
         ...   
495   -0.002821
496    0.008839
497    0.008294
498    0.009616
499   -0.000230
Name: ALGO, Length: 500, dtype: float64
0           NaN
1           NaN
2           NaN
3           NaN
4     -0.006620
         ...   
495   -0.006044
496   -0.008369
497   -0.003086
498    0.004771
499    0.008917
Name: ALGO, Length: 500, dtype: float64


In [26]:
import numpy as np
import pandas as pd

def backtest_momentum_strategy(prices, asset_names, x=20, pos_limit=10000):
    """
    prices: DataFrame, rows = time, columns = asset names
    asset_names: list of assets to test
    x: rolling window length
    pos_limit: max dollar position long or short
    """

    results = {}

    returns = prices[asset_names].pct_change()

    ave_returns = returns.shift(1).rolling(x).mean()
    std_returns = returns.shift(1).rolling(x).std()

    for asset in asset_names:
        cash = 0
        position_units = 0

        portfolio_values = []

        for i in range(len(prices)):
            price = prices[asset].iloc[i]

            # Current portfolio value before new trade
            portfolio_value = cash + position_units * price
            portfolio_values.append(portfolio_value)

            # Skip if signal is not ready
            signal = ave_returns[asset].iloc[i] / std_returns[asset].iloc[i]

            if pd.isna(signal) or pd.isna(price) or std_returns[asset].iloc[i] == 0:
                continue

            # Cap signal between -1 and 1
            signal = max(min(signal, 1), -1)

            # Current dollar exposure
            current_dollar_pos = position_units * price

            # Target dollar exposure from signal
            target_dollar_pos = signal * pos_limit

            # Dollar amount to trade
            trade_dollar = target_dollar_pos - current_dollar_pos

            # Update cash and position
            cash -= trade_dollar
            position_units += trade_dollar / price

        portfolio_values = pd.Series(portfolio_values, index=prices.index)

        # Strategy returns based on changes in portfolio value
        strategy_pnl = portfolio_values.diff()

        # Since pos_limit is our capital/risk scale, divide PnL by pos_limit
        strategy_returns = strategy_pnl / pos_limit

        mean_return = strategy_returns.mean()
        std_return = strategy_returns.std()

        if std_return == 0 or pd.isna(std_return):
            sharpe = np.nan
        else:
            sharpe = mean_return / std_return

        results[asset] = {
            "final_profit": portfolio_values.iloc[-1],
            "sharpe": sharpe,
            "portfolio_values": portfolio_values,
            "strategy_returns": strategy_returns
        }

    return results

In [30]:

asset_names = ['SRNA', 'NPCK', 'HRET', 'HRND', 'ILVX']

results = backtest_momentum_strategy(
    prices=prices,
    asset_names=asset_names,
    x=20,
    pos_limit=10000
)

for asset in asset_names:
    print(asset)
    print("Final profit:", results[asset]["final_profit"])
    print("Sharpe:", results[asset]["sharpe"])
    print()


SRNA
Final profit: 3386.36478338116
Sharpe: 0.1349551436876325

NPCK
Final profit: 1333.466598047606
Sharpe: 0.05010538104426636

HRET
Final profit: 375.18997808827953
Sharpe: 0.015656391120664012

HRND
Final profit: 1150.4230715858757
Sharpe: 0.0335393959433199

ILVX
Final profit: 1163.7581017608807
Sharpe: 0.06084164105601608

